# 12. Скорость обучения. LR_Scheduler

Видео: [12. Скорость обучения. LR_Scheduler](https://www.youtube.com/watch?v=oWevpnv3RI0&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=13)

## О чём урок

В видео 11 lr был **фиксированный** (или автор вручную пересоздал Adam с меньшим lr).  
**Scheduler** делает то же **автоматически** по ходу эпох: `scheduler.step(...)`.

> **Конспект:** [`d.4.2`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.2 Градиентная оптимизация.md) **§2.6** (lr decay, дообучить vs с нуля). Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).

| scheduler | идея |
|:---|:---|
| `StepLR` | каждые `step_size` эпох: `lr *= gamma` |
| `MultiStepLR` | то же, но в выбранные эпохи (`milestones`) |
| `ExponentialLR` | на **каждом** `step()`: `lr *= gamma` |
| `ReduceLROnPlateau` | если метрика/loss **не улучшается** `patience` эпох → `lr *= factor` |

**Важно:**
- `scheduler.step()` вызывают **раз в эпоху** (после train, обычно после val), не на каждом батче;
- у `ReduceLROnPlateau` — `scheduler.step(val_loss)` (или метрика), у остальных — `scheduler.step()` без аргумента;
- читать текущий lr: `opt.param_groups[0]["lr"]`.

**Как смотреть.** ~15 мин — целиком или кусками по таблице schedulers → практика.

**Как решать.** Замени `...` → проверка. Решения — `12_lr_scheduler_решения.ipynb` после попытки.

In [1]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR, MultiStepLR, ExponentialLR, ReduceLROnPlateau

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def current_lr(opt):
    return opt.param_groups[0]["lr"]


print("torch", torch.__version__)

torch 2.14.1+cpu


## 1. <span style="color:#7CDBA0">StepLR</span> — каждые N эпох умножить lr

```python
sched = StepLR(opt, step_size=2, gamma=0.1)
# ... эпоха обучения ...
sched.step()  # раз в эпоху
```

Старт `lr=1.0`, `step_size=2`, `gamma=0.1`:
- эпохи 0–1: lr = 1.0  
- после 2-го `step` (конец эпохи с индексом 1): lr = 0.1  
- после ещё 2 step: lr = 0.01  

Точная нумерация: lr падает, когда число вызовов `step()` стало кратно `step_size`.

In [3]:
# 1.1 StepLR: собери список lr на 6 вызовах step()
opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=1.0)
 # StepLR(opt, step_size=2, gamma=0.1)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.1) 

lrs = [] 
for _ in range(6):
    lrs.append(current_lr(opt))
    sched.step()

print("lrs:", lrs)

lrs: [1.0, 1.0, 0.1, 0.1, 0.010000000000000002, 0.010000000000000002]


C:\Users\HUAWEI\AppData\Local\Temp\ipykernel_6672\4270817392.py:9: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


In [4]:
need(isinstance(sched, StepLR), "sched = StepLR(opt, step_size=2, gamma=0.1)")
need(lrs[0] == 1.0, f"старт lr={lrs[0]}")
need(abs(lrs[2] - 0.1) < 1e-9, f"после 2 step ждём 0.1, а {lrs[2]}")
need(abs(lrs[4] - 0.01) < 1e-9, f"после 4 step ждём 0.01, а {lrs[4]}")
print("1.1 — верно")

1.1 — верно


## 2. <span style="color:#C9A0DC">MultiStepLR</span> и <span style="color:#E6C07B">ExponentialLR</span>

**MultiStepLR** — как StepLR, но падения только в указанные эпохи: `milestones=[2, 5]`.

**ExponentialLR** — на каждом `step()` сразу `lr *= gamma` (плавная «экспонента»).

In [5]:
# 2.1 MultiStepLR + ExponentialLR
w = torch.zeros(1, requires_grad=True)
opt_m = torch.optim.SGD([w], lr=1.0)
opt_e = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=1.0)

sched_m = MultiStepLR(opt_m, milestones=[2, 4], gamma=0.1)
sched_e = ExponentialLR(opt_e, gamma=0.5)

lrs_m, lrs_e = [], []
for _ in range(6):
    lrs_m.append(current_lr(opt_m))
    lrs_e.append(current_lr(opt_e))
    sched_m.step()
    sched_e.step()

print("MultiStep:", lrs_m)
print("Expo:    ", [round(x, 4) for x in lrs_e])

MultiStep: [1.0, 1.0, 0.1, 0.1, 0.010000000000000002, 0.010000000000000002]
Expo:     [1.0, 0.5, 0.25, 0.125, 0.0625, 0.0312]


C:\Users\HUAWEI\AppData\Local\Temp\ipykernel_6672\1369629992.py:13: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched_m.step()
C:\Users\HUAWEI\AppData\Local\Temp\ipykernel_6672\1369629992.py:14: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched_e.step()


In [6]:
need(isinstance(sched_m, MultiStepLR) and isinstance(sched_e, ExponentialLR), "MultiStepLR / ExponentialLR")
need(abs(lrs_m[0] - 1.0) < 1e-9 and abs(lrs_m[2] - 0.1) < 1e-9, f"MultiStep lrs={lrs_m}")
need(abs(lrs_m[4] - 0.01) < 1e-9, f"после milestone 4 ждём 0.01, а {lrs_m[4]}")
need(abs(lrs_e[1] - 0.5) < 1e-9 and abs(lrs_e[2] - 0.25) < 1e-9, f"Expo lrs={lrs_e}")
print("2.1 — верно")

2.1 — верно


## 3. <span style="color:#6CB6FF">ReduceLROnPlateau</span> — самый частый

Как ручное «уменьшил lr после плато» из видео 11, только автоматически.

```python
sched = ReduceLROnPlateau(opt, mode="min", factor=0.1, patience=2)
# после val:
sched.step(val_loss)   # mode="min" — ждём уменьшение loss
```

| аргумент | смысл |
|:---|:---|
| `mode` | `"min"` для loss; `"max"` для accuracy |
| `factor` | на сколько умножить lr при срабатывании (часто 0.1) |
| `patience` | сколько эпох **без значимого улучшения**, потом снизить lr |

Ниже loss искусственно «застыл» — scheduler должен уменьшить lr.

In [7]:
# 3.1 ReduceLROnPlateau: плато → lr падает
opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=1.0)
sched = ReduceLROnPlateau(opt, mode="min", factor=0.1, patience=2)

# loss не улучшается: всё время 1.0
lrs = []
for _ in range(6):
    lrs.append(current_lr(opt))
    sched.step(1.0)

print("lrs:", lrs)

lrs: [1.0, 1.0, 1.0, 1.0, 0.1, 0.1]


In [8]:
need(isinstance(sched, ReduceLROnPlateau), "ReduceLROnPlateau(...)")
need(abs(lrs[0] - 1.0) < 1e-9, f"start={lrs[0]}")
need(min(lrs) < 1.0 - 1e-9, f"на плато lr должен упасть, lrs={lrs}")
need(abs(min(lrs) - 0.1) < 1e-6 or abs(min(lrs) - 0.01) < 1e-6, f"ждали factor=0.1, lrs={lrs}")
print("3.1 — верно")

3.1 — верно


## 4. Вставить scheduler в цикл эпохи

Псевдокод (классификация / регрессия — без разницы):

```python
for epoch in range(EPOCHS):
    train_one_epoch(...)
    val_loss = validate(...)
    sched.step(val_loss)          # ReduceLROnPlateau
    # или: sched.step()           # StepLR / MultiStep / Exponential
```

Собери короткий цикл: 1 шаг Adam на игрушечных данных + `ReduceLROnPlateau`, передавай в `step` одно и то же `val_loss`, чтобы lr снизился.

In [9]:
# 4.1 Мини-цикл: train-шаг + ReduceLROnPlateau(val_loss)
torch.manual_seed(0)
model = nn.Linear(4, 2)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.MSELoss()
sched = ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=1)

xb = torch.randn(8, 4)
yb = torch.randn(8, 2)

lrs = []
for epoch in range(5):
    model.train()
    pred = model(xb)
    loss = criterion(pred, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()

    # «val» застрял на 1.0 — имитация плато
    val_loss = 1.0
    sched.step(val_loss)
    lrs.append(current_lr(opt))

print("lrs:", [round(x, 6) for x in lrs])

lrs: [0.01, 0.01, 0.005, 0.005, 0.0025]


In [10]:
need(isinstance(sched, ReduceLROnPlateau), "нужен ReduceLROnPlateau")
need(len(lrs) == 5, "5 эпох")
need(lrs[-1] < lrs[0], f"lr должен упасть на плато: {lrs}")
print("4.1 — верно")

4.1 — верно


## 5. Что мы сделали

| тема | зачем |
|:---|:---|
| StepLR / MultiStep / Expo | заранее известное расписание lr |
| ReduceLROnPlateau | снизить lr, когда val перестал улучшаться |
| `step` раз в эпоху | куда вставить в цикл из видео 11 |

Дальше: раздел **«Видео 12»** в `project/fashion_mnist.ipynb`. Сохранение лучших весов — видео 13.